
# Exercise 1 — Visual quality control, layer generation with LAYNII, and parcel-wise laminar time courses

In this exercise, we will work with files in `data_Ex1/`:

- `func_MEAN.nii`
- `ln_depths_equivol.nii`
- `rim.nii`
- `schaefer_L_in-func.nii`
- `schaefer_R_in-func.nii`
- `t1_in_func.nii`

We will also use parcel-wise laminar time courses in:

- `data_Ex1/layer_timecourses/`
  - `Layer_run1_1.npy`
  - `Layer_run1_2.npy`
  - `Layer_run1_3.npy`

## Learning goals

By the end of this exercise, you should be able to:

1. inspect the alignment between the anatomical image and the mean functional image,
2. visualize layer maps on top of the anatomical image,
3. visualize parcel maps on top of the anatomical image,
4. regenerate layers from a `rim.nii` file using **LAYNII**,
5. load parcel-wise laminar time courses in Schaefer-400 space,
6. plot the time course from a selected parcel and cortical layer,
7. compute and visualize layer-specific adjacency matrices,
8. compute and interpret a full laminar adjacency matrix across all parcels and layers.


## 1. Setup

Make sure the notebook sits next to the utility file `laminar_ex1_utils.py`.

If you want to regenerate layers, you also need the **LAYNII** executable `LN2_LAYERS`
installed and available. `LN2_LAYERS` generates equi-distant or equi-volume layers from a gray-matter
segmentation (the rim file `rim.nii`).


In [ ]:
from pathlib import Path
from IPython.display import Image, display

import laminar_ex1_utils as ex1

DATA_DIR = Path("data_ex1")
OUT_DIR = Path("results_ex1")
OUT_DIR.mkdir(exist_ok=True)

t1_path = DATA_DIR / "t1_in_func.nii"
func_mean_path = DATA_DIR / "func_MEAN.nii"
layers_path = DATA_DIR / "ln_depths_equivol.nii"
rim_path = DATA_DIR / "rim.nii"
schaefer_l_path = DATA_DIR / "schaefer_L_in-func.nii"
schaefer_r_path = DATA_DIR / "schaefer_R_in-func.nii"

print("Input files:")
for p in [t1_path, func_mean_path, layers_path, rim_path, schaefer_l_path, schaefer_r_path]:
    print(" -", p, "| exists:", p.exists())


## 2. Inspect T1 / functional alignment as a GIF

A fast quality-control step is to flip through slices and compare the structural image
with the mean functional image. The GIF below alternates between the two images for multiple slices.


In [ ]:
gif_path = OUT_DIR / "t1_func_alignment.gif"

ex1.make_alignment_gif(
    t1_path,
    func_mean_path,
    DATA_DIR / "alignment.gif",
    frame_duration=1.5,
    n_slices=24,
    add_checkerboard=True,
)

display(Image(filename=str(gif_path)))
print("Saved:", gif_path)


### Interpretation

When the alignment is good, you should see that the gross anatomy in `t1_in_func.nii`
corresponds well to the contrast boundaries in `func_MEAN.nii`.

Look especially at:
- the cortical outline,
- ventricles,
- major sulci and gyri,
- obvious left/right shifts or rotations.



## 3. Plot the existing layer map on top of T1

Next, we visualize `ln_depths_equivol.nii` as a transparent overlay on top of `t1_in_func.nii`.
This lets us check whether the depth map follows the cortical ribbon in a sensible way. 
This depth map marks the distance of each gray matter voxel from the white matter (the further, the larger the value).


In [ ]:
layer_overlay_png = OUT_DIR / "layers_on_t1.png"

ex1.plot_overlay_on_t1(
    t1_path=t1_path,
    overlay_path=layers_path,
    output_png=layer_overlay_png,
    axis=2,
    alpha=0.5,
    title="Existing equivolume depth map on top of T1",
)

display(Image(filename=str(layer_overlay_png)))
print("Saved:", layer_overlay_png)


## 4. Plot the Schaefer parcels on top of T1

We now repeat the overlay for the parcel maps. Because the left and right parcel images are
separate files, it is often convenient to inspect them one after the other. Here, we use the Schaefer-400 parcellation, a mapping that separates the cortex into 400 areas (200 per hemisphere). 


In [ ]:
schaefer_l_png = OUT_DIR / "schaefer_left_on_t1.png"
schaefer_r_png = OUT_DIR / "schaefer_right_on_t1.png"

ex1.plot_discrete_overlay_on_t1(
    t1_path=t1_path,
    overlay_path=schaefer_l_path,
    output_png=schaefer_l_png,
    axis=2,
    alpha=0.7,
    title="Left Schaefer parcels on top of T1",
)

ex1.plot_discrete_overlay_on_t1(
    t1_path=t1_path,
    overlay_path=schaefer_r_path,
    output_png=schaefer_r_png,
    axis=2,
    alpha=0.7,
    title="Right Schaefer parcels on top of T1",
)

display(Image(filename=str(schaefer_l_png)))
display(Image(filename=str(schaefer_r_png)))


## 5. Regenerate the layers using LAYNII

For this dataset, the key input for layer generation is the **rim file**: `rim.nii`. 

A simple LAYNII command for 3 equi-volume layers is:

```bash
LN2_LAYERS -rim rim.nii -nr_layers 3 -equivol -thickness -output layers_from_rim.nii.gz
```

This is in bash, but here we're using a python implementation of laynii. 

In [ ]:
rim_png = OUT_DIR / "rim_on_t1.png"

ex1.plot_discrete_overlay_on_t1(
    t1_path=t1_path,
    overlay_path=rim_path,
    output_png=rim_png,
    axis=2,
    alpha=0.7,
    title="Rim file on top of T1",
)

display(Image(filename=str(rim_png)))

Here's the rim file: just a segmentation of white and gray matter. 

In [ ]:
regenerated_layers_path = ex1.run_laynii_layers(
    rim_path=rim_path,
    output_dir=OUT_DIR / "laynii_layers",
    n_layers=3,
    equivol=True,
    thickness=True,
    laynii_executable="LN2_LAYERS",
    output_stem="layers_from_rim",
)

print("Saved:", regenerated_layers_path)


## 6. Compare the regenerated layers with the existing layer map

Once you have regenerated the layers, let's take a look at them again. Here we will look at the 3 automatically generated layers rather than the distance file: `layers_from_rim_layers_equidist.nii.gz`.


In [ ]:
regenerated_overlay_png = OUT_DIR / "regenerated_layers_on_t1.png"
regenerated_layers_path = OUT_DIR / "laynii_layers/layers_from_rim_layers_equidist.nii.gz"
ex1.plot_discrete_overlay_on_t1(
    t1_path=t1_path,
    overlay_path=regenerated_layers_path,
    output_png=regenerated_overlay_png,
    axis=1,
    alpha=0.7,
    labels=[1, 2, 3],
    colors=["#3b4cc0", "#78c679", "#fdae61"],
    title="Regenerated LAYNII layers on top of T1",
)

display(Image(filename=str(regenerated_overlay_png)))


## 7. Parcel-wise laminar time courses in Schaefer-400 space

In this final part of Exercise 1, we move from anatomical visualization to parcel-wise laminar signals.

The files in `data_Ex1/layer_timecourses/` contain time courses that have already been extracted in **Schaefer-400 space** for three cortical depths:

- deep
- middle
- superficial

Each file has shape `(n_parcels, n_timepoints)`, so each row corresponds to one Schaefer parcel and each column corresponds to one time point.

We will do three analyses:

1. plot the time course from a chosen parcel in a chosen layer
2. compute a parcel-by-parcel adjacency matrix using Pearson correlation
3. compute a layer-by-layer parcel-by-parcel full adjacency matrix

In [8]:
import matplotlib.pyplot as plt
import numpy as np

from laminar_ex1_utils_connectivity import (
    load_layer_timecourses,
    plot_parcel_timecourse,
    compute_adjacency_matrix,
    plot_adjacency_matrix,
    compute_all_layer_adjacencies,
    compute_full_laminar_adjacency,
    plot_full_laminar_adjacency,
)

### Load the laminar Schaefer time courses

In [ ]:
layer_dir = Path("data_ex1/layer_timecourses")
layer_data = load_layer_timecourses(layer_dir)

for layer_name, arr in layer_data.items():
    print(f"{layer_name:12s} shape = {arr.shape}")

Each array has the shape:

- rows = Schaefer parcels (400)
- columns = time points (138 volumes in this case; approx. 8 min)

### Plot the time course from one parcel and one layer

Let us inspect the signal from one Schaefer parcel in one cortical layer.
You can change both the parcel index and the layer name.

In [ ]:
parcel_idx = 120
layer_name = "superficial"   # choose from: "deep", "middle", "superficial"

plt.figure(figsize=(9, 3.5))
plot_parcel_timecourse(
    layer_data=layer_data,
    parcel_idx=parcel_idx,
    layer_name=layer_name,
    tr=2.0,
)
plt.show()

This plot shows the parcel-average time course for one cortical depth.

Try changing:

- `parcel_idx`
- `layer_name`

to explore how the signal differs across parcels and layers.

### Compute an adjacency matrix for one layer

We now summarize parcel-to-parcel relationships with a correlation-based adjacency matrix.

Each entry in the matrix is the Pearson correlation between two Schaefer parcels within the selected layer.

In [ ]:
layer_name = "superficial"

adj = compute_adjacency_matrix(
    layer_data=layer_data,
    layer_name=layer_name,
    zero_diagonal=True,
)

print("Adjacency shape:", adj.shape)
print("Min value:", np.min(adj))
print("Max value:", np.max(adj))

In [ ]:
plt.figure(figsize=(7, 6))
plot_adjacency_matrix(adj, layer_name=layer_name, vmin=-1, vmax=1)
plt.show()

The heatmap above shows the parcel-by-parcel adjacency matrix for one cortical layer.

A few points to note:

- the matrix is square because every parcel is compared with every other parcel
- warm colors indicate positive correlation
- cool colors indicate negative correlation
- the diagonal is set to zero here

### Compare adjacency matrices across layers

Because we have deep, middle, and superficial time courses, we can compare connectivity patterns across cortical depth.

In [ ]:
adj_dict = compute_all_layer_adjacencies(layer_data, zero_diagonal=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, layer_name in zip(axes, ["deep", "middle", "superficial"]):
    plot_adjacency_matrix(adj_dict[layer_name], layer_name=layer_name, ax=ax, vmin=-1, vmax=1)

plt.tight_layout()
plt.show()

This comparison lets us ask whether parcel-to-parcel connectivity looks similar across depth, or whether some patterns are stronger in one layer than another.

### Full laminar adjacency matrix across all parcels and layers

So far, we have looked at adjacency matrices one layer at a time.

We can also combine all layer-specific parcel time courses into one larger matrix and correlate everything with everything. In this representation:

- each Schaefer parcel appears once in the deep layer
- once in the middle layer
- once in the superficial layer

For Schaefer-400 and 3 layers, this gives:

- `400 × 3 = 1200` layer-specific nodes
- a full adjacency matrix of size `1200 × 1200`

This matrix contains:

- within-layer correlations
- across-layer correlations
- both within the same parcel and between different parcels

In [ ]:
adj_full = compute_full_laminar_adjacency(
    layer_data=layer_data,
    layer_order=["deep", "middle", "superficial"],
    zero_diagonal=True,
)

print("Full laminar adjacency shape:", adj_full.shape)
print("Min value:", np.min(adj_full))
print("Max value:", np.max(adj_full))

In [ ]:
plt.figure(figsize=(8, 8))
plot_full_laminar_adjacency(
    adj_full,
    n_parcels=400,
    layer_names=["deep", "middle", "superficial"],
    vmin=-1,
    vmax=1,
    show_boundaries=True,
)
plt.show()

The black lines divide the matrix into layer-specific blocks.

This means:

- the top-left block shows deep-to-deep correlations
- the middle block shows middle-to-middle correlations
- the bottom-right block shows superficial-to-superficial correlations

The off-diagonal blocks show correlations **between layers**:

- deep ↔ middle
- deep ↔ superficial
- middle ↔ superficial

This gives a more complete view of laminar functional relationships than plotting each layer separately.


## Summary

In this exercise, you:

- checked anatomical / functional alignment with a GIF,
- visualized a layer map on top of the anatomical image,
- visualized Schaefer parcel maps on top of the anatomical image,
- learned how to regenerate layers from a rim file using **LAYNII**,
- loaded parcel-wise laminar time courses in Schaefer-400 space,
- plotted the time course from a selected parcel and cortical layer,
- computed layer-specific adjacency matrices,
- explored a full laminar adjacency matrix across all parcels and layers.

### Practical takeaway

Before moving on to laminar analyses, it is good practice to verify:

1. image alignment,
2. quality of the layer map,
3. quality of the ROI / parcel placement,
4. whether parcel-wise time courses look reasonable,
5. whether connectivity patterns are sensible within and across layers.

These checks help ensure that later laminar analyses are interpretable and based on well-aligned, well-defined data.